# 18b · DMID reference audit (full resolution) + CBIS-DDSM cohort descriptors
**Runtime: CPU** (about 20–40 min; mostly the DMID download). Reads only public data; writes to `results/p2_dmid_audit18b/`.
Does not touch any model, checkpoint or training run, and shows **no model predictions** (audit is blind to model errors).

Outputs:
1. `assignments.csv` – every listed DMID mass: assigned component ids, areas, full-resolution bounding boxes (x0, y0, x1, y1), distance of each component to the listed centre in units of r, nearest unassigned component, components shared with other records, status (retained / duplicate / skipped) and review flags.
2. `duplicates_fullres.csv` – the 19 removed records vs their retained counterpart: exact equality of the full-resolution component-label set and of the binary mask.
3. `sheets/*.png` – overlay sheets for all records (yellow: listed centre and radius r; dashed yellow: 1.5 r assignment limit; red: assigned components with ids; cyan: other components of the image).
4. `audit_template.csv` – one row per record for the reviewer: decision (accept / correct / exclude / uncertain), corrected component ids, reason.
5. `cbis_descriptors.csv` + printed summary – pathology, BI-RADS assessment and subtlety of the 400 selected ROIs vs all 1,696 eligible mass ROIs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, subprocess, requests, json, re
import numpy as np, pandas as pd, cv2, tifffile
from PIL import Image
P = '/content/drive/MyDrive/quantum-placement'
OUT = f'{P}/results/p2_dmid_audit18b'; os.makedirs(f'{OUT}/sheets', exist_ok=True)
DEDUP = f'{P}/results/p2_analysis_v2/dmid_dedup.csv'
JITMETA = f'{P}/data/external/dmid_mass_256_jit/roi/metadata.csv'
RAW, LOCAL = '/content/dmid_raw', '/content/dmid'; os.makedirs(RAW, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)
print(os.path.exists(DEDUP), os.path.exists(JITMETA))

In [ ]:
# 1 · DMID download (figshare 24522883) — same files and parsing as notebook 11
art = requests.get('https://api.figshare.com/v2/articles/24522883', timeout=60).json()
for f in art['files']:
    if not any(k in f['name'] for k in ('TIFF', 'ROI', 'Metadata')): continue
    dst = f"{RAW}/{f['name']}"
    if not os.path.exists(dst) or os.path.getsize(dst) != f['size']:
        with requests.get(f['download_url'], stream=True, timeout=900) as r:
            r.raise_for_status()
            with open(dst, 'wb') as o:
                for ch in r.iter_content(1 << 20): o.write(ch)
    if dst.endswith('.zip') and not os.path.isdir(f"{LOCAL}/{f['name'][:-4]}"):
        subprocess.run(['unzip', '-q', '-o', dst, '-d', LOCAL], check=False)
def read_tif(p):
    try: a = tifffile.imread(p)
    except Exception: a = np.array(Image.open(p))
    a = np.asarray(a)
    if a.ndim == 3 and a.shape[0] in (1, 3, 4) and a.shape[-1] not in (1, 3, 4): a = np.moveaxis(a, 0, -1)
    if a.ndim == 3: a = a[..., :3].mean(-1) if a.shape[-1] >= 3 else a[..., 0]
    return a
def to_uint8(img):
    img = img.astype(np.float32); nz = img[img > 0]
    lo, hi = (np.percentile(nz, 1), np.percentile(nz, 99.5)) if nz.size else (img.min(), img.max())
    return (np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)
info_path = glob.glob(f'{LOCAL}/**/Info.txt', recursive=True)[0]
rows = []
for line in open(info_path, errors='ignore'):
    p = line.split()
    if len(p) >= 3 and p[0].upper().startswith('IMG'):
        def _f(s):
            try: return float(s)
            except ValueError: return np.nan
        rows.append(dict(img=p[0].upper(), tissue=p[1], cls=p[2], sev=p[3] if len(p) > 3 else None,
                         x=_f(p[4]) if len(p) > 6 else np.nan, y=_f(p[5]) if len(p) > 6 else np.nan, r_raw=_f(p[6]) if len(p) > 6 else np.nan))
info = pd.DataFrame(rows)
IMG_DIR = glob.glob(f'{LOCAL}/**/TIFF Images', recursive=True)[0]
ROI_DIR = glob.glob(f'{LOCAL}/**/ROI Masks', recursive=True)[0]
mass = info[info.cls.isin(['CIRC', 'SPIC', 'MISC']) & info.x.notna() & info.y.notna()].copy()
mass['r'] = mass.r_raw.fillna(50).clip(lower=20)
mass = mass.reset_index(drop=True)            # index i -> roi_id IMGxxx_i, exactly as in notebook 11
mass['roi_id'] = [f'{a}_{i}' for i, a in zip(mass.index, mass.img)]
print('listed masses with coordinates:', len(mass), '| images:', mass.img.nunique())

In [ ]:
# 2 · Assignment at full resolution (rule of notebook 11), with full per-component record
dedup = pd.read_csv(DEDUP); keepset = set(dedup.roi_id[dedup.keep == True]); dupset = set(dedup.roi_id[dedup.keep == False])
def bbox_dist(st, x, y):
    x0, y0, w, h = st[:4]; dx = max(x0 - x, 0, x - (x0 + w - 1)); dy = max(y0 - y, 0, y - (y0 + h - 1)); return float(np.hypot(dx, dy))
REC, CACHE = [], {}
for img_id, grp in mass.groupby('img', sort=True):
    f = f'{ROI_DIR}/{img_id}.tif'
    if not os.path.exists(f):
        for _, r in grp.iterrows(): REC.append(dict(roi_id=r.roi_id, img=img_id, cls=r.cls, sev=r.sev, x=r.x, y=r.y, r=r.r, r_missing=pd.isna(r.r_raw), status='skipped: no mask file'))
        continue
    m = read_tif(f) > 0; im = read_tif(f'{IMG_DIR}/{img_id}.tif')
    if im.shape != m.shape: m = cv2.resize(m.astype(np.uint8), (im.shape[1], im.shape[0]), interpolation=cv2.INTER_NEAREST) > 0
    n, lab, st, cen = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=8)
    CACHE[img_id] = (n, lab, st)
    for _, r in grp.iterrows():
        d = {k: bbox_dist(st[k], r.x, r.y) / r.r for k in range(1, n)}
        keep = sorted(k for k, v in d.items() if v <= 1.5)
        others = sorted((v, k) for k, v in d.items() if k not in keep)
        rec = dict(roi_id=r.roi_id, img=img_id, cls=r.cls, sev=r.sev, x=r.x, y=r.y, r=r.r, r_missing=pd.isna(r.r_raw), n_comp_image=n - 1,
                   comp_ids=' '.join(map(str, keep)), n_comp=len(keep),
                   comp_area=' '.join(str(int(st[k][4])) for k in keep),
                   comp_dist_r=' '.join(f'{d[k]:.2f}' for k in keep),
                   nearest_unassigned_dist_r=round(others[0][0], 2) if others else np.nan,
                   nearest_unassigned_id=others[0][1] if others else np.nan)
        if keep:
            xs0 = min(st[k][0] for k in keep); ys0 = min(st[k][1] for k in keep)
            xs1 = max(st[k][0] + st[k][2] - 1 for k in keep); ys1 = max(st[k][1] + st[k][3] - 1 for k in keep)
            rec.update(x0=xs0, y0=ys0, x1=xs1, y1=ys1, centre_inside_union_bbox=bool(xs0 <= r.x <= xs1 and ys0 <= r.y <= ys1))
        rec['status'] = ('retained' if r.roi_id in keepset else 'duplicate (removed)' if r.roi_id in dupset else
                         'skipped: no component within 1.5 r' if not keep else 'unexpected: not in dedup list')
        REC.append(rec)
A = pd.DataFrame(REC)
# components shared between records of the same image
own = {}
for _, r in A[A.n_comp.fillna(0) > 0].iterrows():
    for k in str(r.comp_ids).split(): own.setdefault((r.img, int(k)), []).append(r.roi_id)
A['shared_with'] = [' '.join(sorted({o for k in str(r.comp_ids).split() if r.n_comp > 0 for o in own[(r.img, int(k))]} - {r.roi_id})) if r.n_comp > 0 else '' for _, r in A.iterrows()]
A['shared_with_retained'] = [' '.join(o for o in s.split() if o in keepset) for s in A.shared_with]
def flags(r):
    f = []
    if r.n_comp > 1: f.append('multi-component')
    if r.shared_with_retained and r.status == 'retained': f.append('shares component with another retained record')
    if r.n_comp > 0 and not r.centre_inside_union_bbox: f.append('listed centre outside assigned region')
    if pd.notna(r.nearest_unassigned_dist_r) and r.nearest_unassigned_dist_r <= 2.0: f.append('unassigned component within 2 r')
    if r.r_missing: f.append('radius missing (50 px used)')
    if r.status.startswith('skipped'): f.append('excluded')
    return '; '.join(f)
A['flags'] = A.apply(flags, axis=1)
A.to_csv(f'{OUT}/assignments.csv', index=False)
print(A.status.value_counts().to_string()); print('retained:', (A.status == 'retained').sum())
print('flag counts (retained):'); print(pd.Series([x for f in A[A.status == 'retained'].flags for x in f.split('; ') if x]).value_counts().to_string())

In [ ]:
# 3 · Duplicates at full resolution: exact label-set and binary-mask equality
D = []
for _, r in A[A.status == 'duplicate (removed)'].iterrows():
    n, lab, st = CACHE[r.img]; ks = set(map(int, str(r.comp_ids).split()))
    cands = A[(A.img == r.img) & (A.status == 'retained') & (A.n_comp > 0)]
    best = None
    for _, c in cands.iterrows():
        kc = set(map(int, str(c.comp_ids).split()))
        same_set = kc == ks
        same_mask = bool(np.array_equal(np.isin(lab, list(ks)), np.isin(lab, list(kc))))
        if same_set or best is None: best = (c, same_set, same_mask)
        if same_set: break
    c, same_set, same_mask = best
    D.append(dict(removed=r.roi_id, retained=c.roi_id, img=r.img, removed_ids=r.comp_ids, retained_ids=c.comp_ids, identical_label_set=same_set, identical_binary_mask=same_mask,
                  removed_bbox=f'{r.x0},{r.y0},{r.x1},{r.y1}', retained_bbox=f'{c.x0},{c.y0},{c.x1},{c.y1}',
                  removed_cls_sev=f'{r.cls}/{r.sev}', retained_cls_sev=f'{c.cls}/{c.sev}', removed_centre=f'{r.x:.0f},{r.y:.0f}', retained_centre=f'{c.x:.0f},{c.y:.0f}'))
D = pd.DataFrame(D); D.to_csv(f'{OUT}/duplicates_fullres.csv', index=False)
print(len(D), 'removed records | identical label set:', int(D.identical_label_set.sum()), '| identical binary mask:', int(D.identical_binary_mask.sum()))
print(D[['removed', 'retained', 'identical_label_set', 'identical_binary_mask', 'removed_cls_sev', 'retained_cls_sev']].to_string())

In [ ]:
# 4 · Overlay sheets (blind to model output): 24 panels per sheet
import matplotlib.pyplot as plt
def panel(ax, r):
    im8 = to_uint8(read_tif(f'{IMG_DIR}/{r.img}.tif'))
    H, W = im8.shape
    if r.img in CACHE:
        n, lab, st = CACHE[r.img]
        if lab.shape != im8.shape: lab = cv2.resize(lab.astype(np.int32).astype(np.float32), (W, H), interpolation=cv2.INTER_NEAREST).astype(np.int32)
    else: n, lab, st = 1, np.zeros_like(im8, dtype=np.int32), None
    half = max(2.5 * r.r, 160)
    if r.n_comp and r.n_comp > 0: half = max(half, 0.7 * max(r.x1 - r.x0, r.y1 - r.y0))
    cx, cy = int(r.x), int(r.y); x0, x1 = max(0, int(cx - half)), min(W, int(cx + half)); y0, y1 = max(0, int(cy - half)), min(H, int(cy + half))
    rgb = cv2.cvtColor(im8[y0:y1, x0:x1], cv2.COLOR_GRAY2RGB); s = 360 / max(rgb.shape[:2]); rgb = cv2.resize(rgb, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    sub = cv2.resize(lab[y0:y1, x0:x1].astype(np.float32), (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_NEAREST).astype(np.int32)
    assigned = set(map(int, str(r.comp_ids).split())) if r.n_comp and r.n_comp > 0 else set()
    for k in np.unique(sub):
        if k == 0: continue
        cs, _ = cv2.findContours((sub == k).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(rgb, cs, -1, (255, 40, 40) if k in assigned else (0, 220, 255), 2 if k in assigned else 1)
        yy, xx = np.nonzero(sub == k); cv2.putText(rgb, str(k), (int(xx.mean()), int(yy.mean())), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 40, 40) if k in assigned else (0, 220, 255), 1)
    c = (int((cx - x0) * s), int((cy - y0) * s))
    cv2.circle(rgb, c, max(1, int(r.r * s)), (255, 230, 0), 1); cv2.drawMarker(rgb, c, (255, 230, 0), cv2.MARKER_CROSS, 10, 1)
    for t in np.linspace(0, 2 * np.pi, 48, endpoint=False)[::2]:
        a, b = t, t + np.pi / 48
        p1 = (int(c[0] + 1.5 * r.r * s * np.cos(a)), int(c[1] + 1.5 * r.r * s * np.sin(a))); p2 = (int(c[0] + 1.5 * r.r * s * np.cos(b)), int(c[1] + 1.5 * r.r * s * np.sin(b)))
        cv2.line(rgb, p1, p2, (255, 230, 0), 1)
    ax.imshow(rgb); ax.axis('off')
    ax.set_title(f"{r.roi_id} {r.cls}/{r.sev} ids[{r.comp_ids if r.n_comp else '-'}]\n{r.status}{(' | ' + r.flags) if r.flags else ''}", fontsize=5.2)
order = pd.concat([A[A.status == 'retained'], A[A.status != 'retained']])
for j in range(0, len(order), 24):
    part = order.iloc[j:j + 24]; fig, axs = plt.subplots(4, 6, figsize=(18, 13))
    for a in axs.ravel(): a.axis('off')
    for a, (_, r) in zip(axs.ravel(), part.iterrows()): panel(a, r)
    plt.tight_layout(); fig.savefig(f'{OUT}/sheets/sheet_{j // 24 + 1:02d}.png', dpi=80); plt.close(fig)
T = order[['roi_id', 'img', 'cls', 'sev', 'status', 'comp_ids', 'flags']].copy()
T['sheet'] = [f'sheet_{k // 24 + 1:02d}' for k in range(len(T))]
T['decision (accept/correct/exclude/uncertain)'] = ''; T['corrected comp_ids'] = ''; T['reason'] = ''; T['reviewer'] = ''
T.to_csv(f'{OUT}/audit_template.csv', index=False)
print('sheets:', len(os.listdir(f'{OUT}/sheets')), '| records:', len(T))

In [ ]:
# 5 · CBIS-DDSM descriptors of the 400 selected ROIs vs all eligible mass ROIs (public TCIA CSVs)
URLS = {'train': ['https://www.cancerimagingarchive.net/wp-content/uploads/mass_case_description_train_set.csv',
                  'https://wiki.cancerimagingarchive.net/download/attachments/22516629/mass_case_description_train_set.csv'],
        'test': ['https://www.cancerimagingarchive.net/wp-content/uploads/mass_case_description_test_set.csv',
                 'https://wiki.cancerimagingarchive.net/download/attachments/22516629/mass_case_description_test_set.csv']}
parts = []
for split, us in URLS.items():
    for u in us:
        try:
            r = requests.get(u, timeout=60); r.raise_for_status(); df = pd.read_csv(io.StringIO(r.text)) if False else pd.read_csv(__import__('io').StringIO(r.text))
            df['split'] = 'Training' if split == 'train' else 'Test'; parts.append(df); print('ok', u); break
        except Exception as e: print('failed', u, e)
if len(parts) == 2:
    C = pd.concat(parts, ignore_index=True)
    C['roi_id'] = 'Mass-' + C.split + '_' + C.patient_id + '_' + C['left or right breast'] + '_' + C['image view'] + '_' + C['abnormality id'].astype(str)
    sel = pd.read_csv(f'{P}/data/processed/cbis_mass_256_jit/folds.csv').roi_id
    C['selected'] = C.roi_id.isin(set(sel))
    print('matched selected ROIs:', int(C.selected.sum()), 'of', len(sel), '| CSV mass ROIs:', len(C))
    C[['roi_id', 'patient_id', 'split', 'pathology', 'assessment', 'subtlety', 'mass shape', 'mass margins', 'selected']].to_csv(f'{OUT}/cbis_descriptors.csv', index=False)
    for col in ['pathology', 'assessment']:
        print(pd.crosstab(C[col], C.selected, normalize='columns').round(3).rename(columns={False: 'not selected', True: 'selected'}).to_string())
    print(C.groupby('selected').subtlety.describe().round(2).to_string())
else:
    print('CBIS-DDSM description CSVs not reachable; upload mass_case_description_train_set.csv and _test_set.csv to', OUT)